# Setup — Initialisation du projet
Exécuter ces cellules au début de chaque session pour monter Google Drive et accéder aux données.

Les constantes (chemins, paramètres audio/STFT) sont définies dans `src/config.py` pour que tout le monde travaille avec les mêmes valeurs.

In [ ]:
# Dépendances Colab — versions épinglées pour reproductibilité
# (librosa, soundfile, wandb sont déjà préinstallés sur Colab, on force juste la version)
%pip install -q librosa==0.11.0 soundfile==0.13.1 wandb==0.26.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Récupération du code du repo (privé)

**Avant la première utilisation**, créer un Personal Access Token GitHub : [github.com/settings/tokens](https://github.com/settings/tokens) → *Generate new token (classic)* → scope `repo` → copier la valeur.

**Récupération du token au runtime** : 3 méthodes essayées dans l'ordre, on prend la première qui marche :
1. Variable d'environnement `GITHUB_TOKEN` (si tu l'as exportée avant le lancement)
2. **Colab Secrets** (panneau 🔑 à gauche, secret `GITHUB_TOKEN` + *Notebook access* activé) — ne marche **que dans l'UI Colab navigateur**, pas depuis VS Code
3. **`getpass`** — la cellule te demande de coller le token (masqué). Fallback universel qui marche partout.

**À chaque session** : changer `BRANCH` ci-dessous pour ta branche de travail.

In [ ]:
import os, sys, subprocess, getpass

# >>> À configurer selon ta branche <<<
BRANCH = 'main'

REPO_OWNER = 'Theo-Lempereur'
REPO_NAME = 'Filtre-Voix-DL'
REPO_DIR = f'/content/{REPO_NAME}'


def _get_github_token():
    # 1) variable d'environnement
    tok = os.environ.get('GITHUB_TOKEN')
    if tok:
        print("Token : variable d'environnement")
        return tok
    # 2) Colab Secrets (UI Colab navigateur uniquement)
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
        if tok:
            print('Token : Colab Secrets')
            return tok
    except Exception:
        pass
    # 3) saisie manuelle masquée
    print('Token : saisie manuelle (getpass)')
    return getpass.getpass('Colle ton GitHub Personal Access Token : ')


token = _get_github_token()
repo_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, repo_url, REPO_DIR], check=True)
else:
    # Mise à jour de l'origin (au cas où le token aurait changé)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', repo_url], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Affiche la branche et le dernier commit pour vérification
current = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
last_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip()
print(f'Branche : {current}')
print(f'HEAD    : {last_commit}')

### Auto-reload
Recharge automatiquement les modules `src/` quand tu fais `!git pull` en cours de session — pas besoin de redémarrer le kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src import config

# Création des dossiers Drive s'ils n'existent pas
for path in config.ALL_DIRS:
    os.makedirs(path, exist_ok=True)
    print(f'OK {path}')

print(f'\nSample rate : {config.SAMPLE_RATE} Hz | clip : {config.CLIP_DURATION}s | n_fft={config.N_FFT}, hop={config.HOP_LENGTH}')
print('Drive prêt !')

### Vérification
Affiche le contenu du dossier Drive pour confirmer que tout est en place.

In [ ]:
for root, dirs, files in os.walk(config.DRIVE_PROJECT):
    level = root.replace(config.DRIVE_PROJECT, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    print(f'{indent}{folder_name}/')
    for f in files:
        print(f'{indent}  {f}')